# 🏙️ Kensei Challenge — Inteligência de Mercado Airbnb Rio de Janeiro

Análise automatizada do mercado Airbnb usando dados públicos do [InsideAirbnb](https://insideairbnb.com/rio-de-janeiro/) + Claude AI.

**Fluxo:**
1. Coleta automática dos dados mais recentes
2. Limpeza e enriquecimento
3. Análises por bairro, tipo, preço, ocupação
4. Visualizações
5. Insights gerados por IA (Claude)
6. Exportação do relatório em Markdown

## 0. Instalação de Dependências

In [ ]:
# Execute este bloco apenas na primeira vez
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'])
print('✅ Dependências instaladas')

## 1. Imports e Configuração

In [ ]:
import os, warnings
from pathlib import Path
from IPython.display import display, Markdown

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_palette(['#FF5A5F','#FC642D','#00A699','#484848','#767676'])

# Importa funções do script principal
from kensei_airbnb import (
    discover_latest_date, load_listings, load_reviews,
    clean_listings, Analyzer, AIInsights, generate_report,
    DATA_DIR, CHARTS_DIR, REPORTS_DIR
)

print(f'ANTHROPIC_API_KEY configurada: {bool(os.getenv("ANTHROPIC_API_KEY"))}')

## 2. Coleta de Dados

In [ ]:
date = discover_latest_date()
print(f'Data dos dados: {date}')

In [ ]:
df_raw = load_listings(date)
df_raw.head(3)

## 3. Limpeza e Enriquecimento

In [ ]:
df = clean_listings(df_raw)
print(f'Shape final: {df.shape}')
df[['price','est_occupancy','est_monthly_revenue']].describe()

## 4. Análises

In [ ]:
analyzer = Analyzer(df)
results = analyzer.run()

### 4.1 Visão Geral

In [ ]:
ov = results['overview']
display(Markdown(f"""
| Métrica | Valor |
|---|---|
| Listings | {ov['total_listings']:,} |
| Preço Mediano | R$ {ov['median_price']:,.0f}/noite |
| Preço Médio | R$ {ov['mean_price']:,.0f}/noite |
| Faixa P25–P75 | R$ {ov['p25_price']:,.0f} – R$ {ov['p75_price']:,.0f} |
| Bairros | {ov.get('total_bairros','?')} |
| % Superhosts | {ov.get('superhost_pct',0):.1f}% |
| Nota Média | {ov.get('avg_rating',0):.2f} / 5.0 |
| Ocupação Estimada | {ov.get('avg_occupancy_pct',0):.1f}% |
| Receita Mediana/Mês | R$ {ov.get('median_monthly_revenue',0):,.0f} |
"""))

### 4.2 Distribuição de Preços

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuição de Preços — Airbnb Rio de Janeiro', fontsize=14, fontweight='bold')

prices = df['price'].dropna()
axes[0].hist(prices, bins=60, color='#FF5A5F', alpha=0.8, edgecolor='white', linewidth=0.3)
axes[0].axvline(prices.median(), color='#484848', ls='--', lw=2, label=f'Mediana R${prices.median():.0f}')
axes[0].axvline(prices.mean(), color='#00A699', ls='--', lw=2, label=f'Média R${prices.mean():.0f}')
axes[0].set_xlabel('Preço (R$/noite)'); axes[0].set_ylabel('Frequência')
axes[0].set_title('Histograma'); axes[0].legend()

if 'room_type' in df.columns:
    rts = df['room_type'].unique()
    room_prices = [df[df['room_type']==rt]['price'].dropna() for rt in rts]
    axes[1].boxplot(room_prices, labels=rts, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    axes[1].set_title('Boxplot por Tipo'); axes[1].set_ylabel('R$/noite')
    plt.setp(axes[1].get_xticklabels(), rotation=15, ha='right')
plt.tight_layout(); plt.show()

### 4.3 Preço por Bairro

In [ ]:
pn = results['price_neighborhood']
display(pn.head(20).style.format({'mediana':'R${:.0f}','media':'R${:.0f}','total':'{:,.0f}','dp':'R${:.0f}'}))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
data = pn.head(20)
bars = ax.barh(range(len(data)), data['mediana'], color='#FF5A5F', alpha=0.85)
ax.set_yticks(range(len(data)))
ax.set_yticklabels(data.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Preço Mediano (R$/noite)')
ax.set_title('Top 20 Bairros por Preço Mediano', fontsize=13, fontweight='bold')
for bar, (_, row) in zip(bars, data.iterrows()):
    ax.text(bar.get_width()+8, bar.get_y()+bar.get_height()/2,
            f'R${bar.get_width():.0f}  ({int(row["total"])})', va='center', fontsize=8)
plt.tight_layout(); plt.show()

### 4.4 Ocupação e Receita por Bairro

In [ ]:
av = results['availability']
occ = av.get('occ_by_bairro', {})
rev = av.get('rev_by_bairro', {})

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ocupação
if occ:
    nbhds_o = list(occ.keys())[:15]
    vals_o = [occ[n] for n in nbhds_o]
    colors_o = ['#FF5A5F' if v > 70 else '#FC642D' if v > 50 else '#00A699' for v in vals_o]
    bars_o = axes[0].barh(range(len(nbhds_o)), vals_o, color=colors_o, alpha=0.85)
    axes[0].set_yticks(range(len(nbhds_o))); axes[0].set_yticklabels(nbhds_o, fontsize=9)
    axes[0].invert_yaxis()
    axes[0].axvline(50, color='#484848', ls='--', lw=1.5)
    axes[0].set_xlabel('Taxa de Ocupação Estimada (%)')
    axes[0].set_title('Top 15: Ocupação Estimada', fontweight='bold')
    for bar in bars_o:
        axes[0].text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                     f'{bar.get_width():.1f}%', va='center', fontsize=8)

# Receita
if rev:
    nbhds_r = list(rev.keys())[:15]
    vals_r = [rev[n] for n in nbhds_r]
    bars_r = axes[1].barh(range(len(nbhds_r)), vals_r, color='#FC642D', alpha=0.85)
    axes[1].set_yticks(range(len(nbhds_r))); axes[1].set_yticklabels(nbhds_r, fontsize=9)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Receita Mensal Estimada (R$)')
    axes[1].set_title('Top 15: Receita Estimada', fontweight='bold')
    for bar in bars_r:
        axes[1].text(bar.get_width()+50, bar.get_y()+bar.get_height()/2,
                     f'R${bar.get_width():,.0f}', va='center', fontsize=8)

plt.tight_layout(); plt.show()

### 4.5 Preço × Reviews

In [ ]:
rv = results['reviews']
print(f"Nota média: {rv.get('avg_rating',0):.2f} | "
      f"% ≥ 4.5: {rv.get('pct_gt_4_5',0):.1f}% | "
      f"Correlação preço×reviews: {rv.get('price_reviews_corr',0):.3f}")

if {'price','number_of_reviews','review_scores_rating'}.issubset(df.columns):
    sample = df[['price','number_of_reviews','review_scores_rating']].dropna().sample(
        min(2500, len(df)), random_state=42)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sc = ax.scatter(sample['number_of_reviews'], sample['price'],
                    c=sample['review_scores_rating'], cmap='RdYlGn',
                    alpha=0.45, s=18, vmin=3, vmax=5)
    plt.colorbar(sc, ax=ax, label='Nota Média')
    z = np.polyfit(sample['number_of_reviews'], sample['price'], 1)
    xl = np.linspace(0, sample['number_of_reviews'].quantile(0.95), 100)
    ax.plot(xl, np.poly1d(z)(xl), 'r--', alpha=0.7, lw=1.5, label='Tendência')
    ax.set_xlabel('Número de Reviews'); ax.set_ylabel('Preço (R$/noite)')
    ax.set_title('Preço × Reviews', fontsize=13, fontweight='bold')
    ax.legend(); plt.tight_layout(); plt.show()

### 4.6 Saturação de Mercado

In [ ]:
sat = results['saturation']
if not sat.empty:
    display(sat.head(10).style.format({
        'median_price': 'R${:.0f}',
        'avg_occupancy': '{:.1%}',
        'superhost_pct': '{:.1%}',
        'saturation_score': '{:.2f}',
    }))

## 5. Insights com Claude AI

In [ ]:
if os.getenv('ANTHROPIC_API_KEY'):
    ai = AIInsights(results)
    insights = ai.generate()
else:
    print('⚠️  Configure ANTHROPIC_API_KEY no .env para gerar insights de IA')
    insights = {k: {'title': t, 'content': '_Sem API key_'}
                for k, t in [('hospedes','Hóspedes'),('anfitrioes','Anfitriões'),('mercado','Mercado')]}

In [ ]:
for key in ['hospedes', 'anfitrioes', 'mercado']:
    ins = insights.get(key, {})
    icons = {'hospedes': '🧳', 'anfitrioes': '🏠', 'mercado': '📈'}
    display(Markdown(f"## {icons[key]} {ins.get('title', key)}\n\n{ins.get('content', '')}"))
    print('\n' + '─'*60 + '\n')

## 6. Exportar Relatório

In [ ]:
report_path = generate_report(results, insights, date)
print(f'Relatório salvo em: {report_path}')

# Exibe o relatório no notebook
with open(report_path, encoding='utf-8') as f:
    display(Markdown(f.read()))